Build Deep Sequential Models & Training it from Scratch
=======================================================

# Setup

In [ ]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
from copy import deepcopy
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import kagglehub
import datasets
import torch.nn as nn
import tokenizers
import transformers
import pandas as pd
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

torch.manual_seed(42)

In [ ]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Setup W&B for experiment tracking

In [ ]:
import wandb

wandb.login(key="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")

# Prepare the data 

## Get the data

In [ ]:
data_path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge",
                                        "train.csv",
                                        output_dir="../data"
                                        )
dataset = datasets.load_dataset("csv", data_files=data_path)
dataset

## Text Normalization

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
url_pattern = re.compile(r'http\S+|www\.\S+')  # Added URL pattern
whitespace_pattern = re.compile(r'\s+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        # Handle None or empty input
        if not text or not text.strip():
            cleaned_texts.append("[UNK]")  # Placeholder for empty texts
            continue
        
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)
        
        # Replace URLs with [UNK]
        text = url_pattern.sub("[UNK]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        # ✅ Ensure text is not empty after cleaning
        if not text:
            text = "[UNK]"  # Fallback for texts that become empty

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
dataset = dataset["train"].map(preprocess_batch, batched=True, num_proc=4)

## Tokenization

### Train a BBPE from scratch

#### Prepare the comments for training

In [ ]:
train = dataset.train_test_split(test_size=0.2, seed=42)["train"]
comments = [comment['comment_text'] for comment in train]
comments[:2]

#### Train the tokenizer

In [ ]:
special_tokens = ['<pad>', '<unk>']


bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=30_000,
                                              special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(comments, bbpe_trainer)

bbpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

#### Wrap our tokenizer in a HuggingFace PreTrainedTokenizerFast for easy integration

In [ ]:
tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bbpe_tokenizer
)

#### Testing our tokenizer

In [ ]:
encodings = tokenizer(comments[:2], truncation=True, max_length=256)

encodings

In [ ]:
encodings["input_ids"]

#### Use our tokenizer

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=256,
        padding=False
    )

In [ ]:
dataset = dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)
dataset = dataset.remove_columns(["comment_text", "id"])

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

## Split data

In [ ]:
from transformers import DataCollatorWithPadding


LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Create a data collator that handles padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def collate_fn(batch):
    # Extract text features for padding
    text_features = [{
        "input_ids": item["input_ids"],
        "attention_mask": item["attention_mask"]
    } for item in batch]
    
    # Pad the batch
    padded = data_collator(text_features)
    
    # Extract labels
    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])
    
    X = {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"]
    }
    
    return X, labels

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_set = split["train"]
test_valid_set = split["test"]
split = test_valid_set.train_test_split(test_size=0.5, seed=42)
valid_set = split["train"]
test_set = split["test"]

### Splits statistics

In [ ]:
print(f"Train Shape: {train_set.shape}")
print(f"Valid Shape: {valid_set.shape}")
print(f"Test Shape: {test_set.shape}")
print("------------------------------------------------")

def calculate_label_statistics(dataset, split_name):
    """Calculate label statistics for a given dataset split."""
    
    # Convert to pandas DataFrame for easier manipulation
    df = dataset.to_pandas()
    
    # Extract label columns
    label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    labels = df[label_cols]
    
    # Calculate statistics
    stats = {}
    
    # 1. Number of rows without any toxic label (all labels are 0)
    stats['no_toxic_labels'] = (labels.sum(axis=1) == 0).sum()
    
    # 2. Number of rows with at least one toxic label
    stats['at_least_one_toxic'] = (labels.sum(axis=1) > 0).sum()
    
    # 3. For each label: number of rows where it takes value 1
    for col in label_cols:
        stats[f'{col}_count'] = labels[col].sum()
    
    # Add total rows for reference
    stats['total_rows'] = len(df)
    
    return stats

# Calculate statistics for each split
train_stats = calculate_label_statistics(train_set, "Train")
valid_stats = calculate_label_statistics(valid_set, "Valid")
test_stats = calculate_label_statistics(test_set, "Test")

# Create a summary DataFrame
summary_stats = pd.DataFrame({
    'Train': train_stats,
    'Valid': valid_stats, 
    'Test': test_stats
}).T

# Display the results
print("=== Label Statistics Summary ===")
print(summary_stats)

# Calculate percentages for better understanding
print("\n=== Label Distribution Percentages ===")
percentage_stats = summary_stats.copy()
for split in ['Train', 'Valid', 'Test']:
    total = summary_stats.loc[split, 'total_rows']
    percentage_stats.loc[split, 'no_toxic_labels_pct'] = (summary_stats.loc[split, 'no_toxic_labels'] / total) * 100
    percentage_stats.loc[split, 'at_least_one_toxic_pct'] = (summary_stats.loc[split, 'at_least_one_toxic'] / total) * 100
    
    for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']:
        percentage_stats.loc[split, f'{col}_pct'] = (summary_stats.loc[split, f'{col}_count'] / total) * 100

# Display percentage columns
percentage_cols = ['no_toxic_labels_pct', 'at_least_one_toxic_pct'] + [f'{col}_pct' for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
print(percentage_stats[percentage_cols].round(2))

VOCAB_SIZE = len(tokenizer)
print(f"Vocabulary size: {VOCAB_SIZE}")


### Positive class weights

In [ ]:
# compute per-label positive/negative counts on the training split
df_train = train_set.to_pandas()
label_counts = df_train[LABEL_COLUMNS].sum()
total_examples = len(df_train)

neg_counts = total_examples - label_counts

# pos_weight is commonly used with BCEWithLogitsLoss for multi-label tasks
pos_weight = (neg_counts / (label_counts + 1e-9)).astype("float32")

print("Label counts (pos):")
print(label_counts)
print("\nPos weights (neg/pos):")
print(pos_weight)

# store as torch tensor for passing to loss
pos_weight_tensor = torch.tensor(pos_weight.values, dtype=torch.float32).to(device)
pos_weight_tensor

## Load the datasets into PyTorch DataLoaders

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Add after creating dataloaders
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch

print("Input shapes:")
print(f"  input_ids: {X_sample['input_ids'].shape}")
print(f"  attention_mask: {X_sample['attention_mask'].shape}")
print(f"  labels: {y_sample.shape}")
print(f"  labels dtype: {y_sample.dtype}")

assert X_sample['input_ids'].shape[0] == BATCH_SIZE
assert y_sample.shape == (BATCH_SIZE, 6)
assert y_sample.dtype == torch.float32

# Train & Eval functions

In [ ]:
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

def evaluate_tm(model, data_loader, metric):
    """Evaluate model on a dataset."""
    
    model.eval()
    metric.reset()

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            metric.update(probs.detach(), y_batch.int())

    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    train_metric,
    val_metric,
    train_loader,
    valid_loader,
    n_epochs,
    patience=2,
    factor=0.5,
    max_grad_norm=1.0,
    checkpoint_path="best_model.pth",
):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        patience=patience,
        factor=factor
    )

    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }

    best_val_metric = 0.0

    for epoch in range(n_epochs):

        model.train()
        train_metric.reset()

        total_loss = 0.0

        for index, (X_batch, y_batch) in enumerate(train_loader):

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = loss_fn(logits, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)

            train_metric.update(
                probs.detach(),
                y_batch.int()
            )

            print(
                f"\rBatch {index+1}/{len(train_loader)} "
                f"loss={total_loss/(index+1):.4f}",
                end=""
            )

        # ----- Epoch metrics -----

        train_metric_value = train_metric.compute().item()
        val_metric_value = evaluate_tm(model, valid_loader, val_metric).item()

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric_value)
        history["valid_metrics"].append(val_metric_value)

        scheduler.step(val_metric_value)

        if val_metric_value > best_val_metric:

            best_val_metric = val_metric_value

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metric": val_metric_value,
                    "history": history,
                },
                checkpoint_path,
            )

        print(
            f"\rEpoch {epoch+1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {train_metric_value:.2%}, "
            f"valid metric: {val_metric_value:.2%} "
            f"{'[BEST]' if val_metric_value == best_val_metric else ''}"
        )

        wandb.log({
            "train/loss": history['train_losses'][-1],
            "train/pr_auc": train_metric_value,
            "val/pr_auc": val_metric_value,
            "epoch": epoch+1/n_epochs
        })

    return history

# Model Building, Training & Evaluation

## Simple Stacked LSTM

In [ ]:
class SimpleStackedLSTMModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=2,
                 hidden_dim=128, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)                      
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),      
                                      batch_first=True, enforce_sorted=False) 
        _outputs, (h_n, c_n) = self.lstm(packed) 
        return self.output(h_n[-1])

### Base Architecture & Hyperparameters

In [ ]:
base_config = {
    # Model architecture
    "model_type": "LSTM",
    "embed_dim": 256,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    
    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "epochs": 10,
    "optimizer": "NAdam",
}

### Config_1 Training

In [ ]:
# Setup New Config
config = base_config

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-1",
                config=config,
                notes="base config for simple stacked lstm model"
               ):
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

**Ammmmm it seems like our model is overfiting the train data**

### Config_2 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 64

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-2",
                config=config,
                notes="changed hidden_dim to 64"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

### Config_3 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 64
config["dropout"] = 0.3

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-3",
                config=config,
                notes="changed hidden_dim + dropout values"
               ):
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

### Config_4 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 32

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-4",
                config=config,
                notes="changed hidden_dim to 32"
               ):

    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

In [ ]:
# Train it more 10 epochs
config["epochs"] = 20

with wandb.init(project="nontoxic-world",
                id="nzdtqh0a",
                resume="allow"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

### Config_5 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["optimizer"] = "AdamW"

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-5",
                config=config,
                notes="changed optimizer to AdamW"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

## Config_6 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-6",
                config=config,
                notes="use pos class weights for the loss function"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

## Config_7 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["hidden_dim"] = 64

model = SimpleStackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)


# Training configuration
n_epochs = config["epochs"]
xentropy = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.NAdam(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-simple-stacked-lstm-config-7",
                config=config,
                notes="use pos class weights + hidden_dim=64"
               ):
    
    history = train(
        model, 
        optimizer, 
        xentropy, 
        macro_pr_auc,
        macro_pr_auc,
        train_loader, 
        valid_loader, 
        n_epochs,
        max_grad_norm=1.0,
        checkpoint_path='simple_stacked_lstm_model_best.pth'
    )

#### Evaluate Each Label seperatly

In [ ]:
val_metric = MultilabelAveragePrecision(
    num_labels=6,
    average=None
).to(device)

results = evaluate_tm(model, valid_loader, val_metric)
print(results)

In [ ]:
train_metric = MultilabelAveragePrecision(
    num_labels=6,
    average=None
).to(device)

results = evaluate_tm(model, train_loader, train_metric)
print(results)